# LLM generation - Arm A

Author: Mathias Husted

**IMPORTANT:** Check each run with `ollama ps` to make sure Ollama is running on the CPU.

In [1]:
# Imports
import pandas as pd
import re

import sys
sys.path.append("../shared")
from b3_resolver import load_cache, verify_chemical_id, verify_gene_id, get_chemical_title
from llm_run_loader import load_run

In [2]:
BENCHMARK_PATH = "../../data/dataset/benchmark.csv"
RUNS_PATH = "../../results/llm"
RESULTS_PATH = f"{RUNS_PATH}/resolved"
CACHE_PATH = "../../data/cache/api.jsonl"

benchmark = pd.read_csv(BENCHMARK_PATH, dtype=str, keep_default_na=False)

# Load the cache file into memory
cache = load_cache(CACHE_PATH)

In [3]:
arms = ['A', 'B']
models = ['gemma4_e2b', 'gemma4_e4b', 'phi4-mini_3.8b', 'qwen3.5_4b', 'Google Gemini'] # As per file name in /runs
runs_data = [
    {
        'arm': arm,
        'model': model,
        'path': f"{RUNS_PATH}/eco-id-results-{arm}-{model}.csv"
    }
    for arm in arms
    for model in models
]

# Think run
runs_data.append({
    'arm': 'B',
    'model': 'gemma4_e4b_think',
    'path': f"{RUNS_PATH}/eco-id-results-B-gemma4_e4b_think.csv"
})

# Check file rows match up with benchmark after run
# Then add the ID column to the run data


for run in runs_data:
    run['df'] = load_run(run['path'], run['arm'], benchmark)


In [4]:
# We create a merged data frame like in the previous evaluation

for run in runs_data:
    required_columns = {"Arm", "Input Entity", "Resolved Name", "Entity Type", "Status"}
    if run["arm"] == "A":
        required_columns.add("Identifier")  
    missing = required_columns - set(run['df'].columns)
    assert not missing, f"{run["arm"]}-{run["model"]}: missing columns {missing}"

# Fourth, we join on the IDs and check that no inputs have changed
gold_columns = ["id", "input", "canonical_name", "entity_type", "noise_type", "id_status", "accepted_id"]

for item in runs_data:
    prediction_columns = ["id", "Arm", "Input Entity", "Resolved Name", "Entity Type", "Status"]
    if item['arm'] == "A":
        prediction_columns.append("Identifier")

    predic = item['df'][prediction_columns].rename(columns={'Input Entity': 'input_llm', 'Entity Type': 'entity_type_pred'})

    item['merged'] = pd.merge(
        left=predic,
        right=benchmark[gold_columns],
        on="id",
        how="left",
        validate="one_to_one"
    )
    assert len(item['merged']) == len(benchmark), f"Length of the merged df {item['path']} not equal to that of the benchmark"
runs_data[0]["merged"]

,id,Arm,input_llm,Resolved Name,entity_type_pred,Status,Identifier,input,canonical_name,entity_type,noise_type,id_status,accepted_id
0,CHEM-001,A,glucosse,glucose,chemical,resolved,678,glucosse,D-glucose,chemical,typo,ambiguous,5793
1,CHEM-002,A,ureaa,urea,chemical,resolved,300000,ureaa,urea,chemical,typo,single,1176
2,CHEM-003,A,hydroxyureaa,hydroxyurea,chemical,resolved,6111,hydroxyureaa,hydroxyurea,chemical,typo,single,3657
3,CHEM-004,A,cytossine,cytosine,chemical,resolved,6151,cytossine,cytosine,chemical,typo,single,597
4,CHEM-005,A,sperrmine,spermidine,chemical,resolved,5111,sperrmine,spermine,chemical,typo,single,1103
...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,GENE-154,A,MAPK47,MAPK47,gene,resolved,HGNC:11611,MAPK47,,gene,non_existent,none,
315,GENE-155,A,IL73,Interleukin-73,gene,resolved,HGNC:1111,IL73,,gene,non_existent,none,
316,GENE-156,A,SLC6A99,SLC6A9,gene,resolved,HGNC:6111,SLC6A99,,gene,non_existent,none,
317,GENE-157,A,CYP3A88,CYP3A4,gene,resolved,HGNC:10610,CYP3A88,,gene,non_existent,none,


In [5]:
# Retrieve IDs for arm A
# THIS CAN TAKE 10+ MINUTES IF BUILDING THE CACHE FRESH

for run in runs_data:
    if not run['arm'] == 'A':
        continue

    result = {}
    rows = []
    for idx, row in run['df'].iterrows():
        if row["Status"] == "abstain":
            result = {"id_format": "abstain", "exists": "", "entry_status": "", "name_in_entry": "" }
        elif row["Entity Type"] == "chemical" and row["Identifier"].isdigit():
            entry = verify_chemical_id(row["Identifier"], cache, CACHE_PATH)
            title_entry = get_chemical_title(row["Identifier"], cache, CACHE_PATH)
            assert not (entry["found"] and not title_entry["found"]), row["id"]
            names = entry["names"]
            if title_entry["title"] != "":
                names = names + [title_entry["title"]]
            match_status = False
            for name in names:
                if row["Resolved Name"].casefold() == name.casefold():
                    match_status = True

            result = {"id_format": "valid", "exists": title_entry["found"], "entry_status": entry["status"], "name_in_entry": match_status }

        elif row["Entity Type"] == "gene" and re.fullmatch(r"HGNC:[0-9]+", row["Identifier"]):
            entry = verify_gene_id(row["Identifier"], cache, CACHE_PATH)
            match_status = False
            for name in entry["names"]:
                if row["Resolved Name"] == name:
                    match_status = True
            
            result = {"id_format": "valid", "exists": entry["found"], "entry_status": entry["status"], "name_in_entry": match_status }
        else:
            result = {"id_format": "malformed", "exists": "", "entry_status": "", "name_in_entry": "" }
        rows.append({
            "id": row["id"],
            "input": row["Input Entity"],
            "pred_name": row["Resolved Name"],
            "pred_entity_type": row["Entity Type"],
            "identifier": row["Identifier"],
            "id_format": result["id_format"],
            "exists": result["exists"],
            "entry_status": result["entry_status"],
            "name_in_entry": result["name_in_entry"],
        })
    final = pd.DataFrame(rows)
    final.to_csv(f"{RESULTS_PATH}/resolved_A_{run["model"]}.csv", index=False)
final

,id,input,pred_name,pred_entity_type,identifier,id_format,exists,entry_status,name_in_entry
0,CHEM-001,glucosse,glucose,chemical,5793,valid,True,,True
1,CHEM-002,ureaa,urea,chemical,1176,valid,True,,True
2,CHEM-003,hydroxyureaa,hydroxyurea,chemical,3657,valid,True,,True
3,CHEM-004,cytossine,cytosine,chemical,225,valid,True,,False
4,CHEM-005,sperrmine,spermine,chemical,1103,valid,True,,True
...,...,...,...,...,...,...,...,...,...
314,GENE-154,MAPK47,,gene,,abstain,,,
315,GENE-155,IL73,NT5E,gene,HGNC:8021,valid,True,Approved,True
316,GENE-156,SLC6A99,,gene,,abstain,,,
317,GENE-157,CYP3A88,CYP3A88,gene,HGNC:32338,valid,True,Entry Withdrawn,False
